# Finding different mathematical functions to model K curve

This is for 1.125P(Vanilla) - P(Cherry) > 0

## Imports

In [147]:
import pickle
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import csv
from math import sqrt, log, e, pi
from statistics import NormalDist

## Import Data

In [96]:
with open('1.125_results.pkl', 'rb') as f:
    results = pickle.load(f)

In [97]:
# x coordinates
x_data = np.array([x for x in range(0, 202)])
# 2D array of all the K_js - y coordinates
y_datas = []
for i in range(len(results)):
    y_datas.append(results[i][0])


## Plot Functions

In [98]:
def plotPolyFig(x_data, y_data, poly_func, poly_pred, degree, alpha):
    fig = plt.figure()
    ax = fig.add_subplot(1,1,1)
    
    ax.plot(y_data, label="Numerical Optimisation", color="blue")
    ax.plot(poly_pred, label=f"{poly_func}", color="red")
    fig.suptitle(f"K curve Polynomial Fit with degree {degree} for $α$ = {alpha} ")
    ax.set_xlabel("j")
    ax.set_ylabel(r"$K_{j}$")
    plt.legend()
    
    plt.savefig(f"plots/fitting/poly_k_curve_alpha_{alpha}_degree_{degree}.png")


## Polynomial fit

In [99]:
def poly(x_data, y_data, degree):
    # Fit the data
    coeffs = np.polyfit(x_data, y_data, degree)
    poly_func = np.poly1d(coeffs)
    
    # get predictions
    poly_pred = poly_func(x_data)
    
    return poly_func, poly_pred
    

## Plot Polynomial

In [100]:
def plotPoly(x_data, y_datas, degree):
    alphas = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
    
    for i, y_data in enumerate(y_datas):

        poly_func, poly_pred = poly(x_data, y_data, degree)
        plotPolyFig(x_data, y_data, poly_func, poly_pred, degree, alphas[i])


In [ ]:
plotPoly(x_data, y_datas, 4)

## Evaluate which fit is the best

### Get MSE of each fit

In [113]:
def evalPoly(x_data, y_datas, degrees):
    alphas = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
    # 2D array of size 9 - first column the degree and the next 8 columns are the MSE for each alpha
    mseTable = []
    
    # for each degree we fit
    for degree in degrees:
        mseTable.append([degree])
        # for each alpha
        for i, y_data in enumerate(y_datas):
            # get the polynomial fit
            poly_func, poly_pred = poly(x_data, y_data, degree)
            # get the mean squared error - round to 4 decimal places
            mse = round(mean_squared_error(y_data, poly_pred), 4)
            # append to the table
            mseTable[-1].append(mse)
            
    return mseTable

In [116]:
degrees = [2, 3, 4, 5, 6]
mseTable = evalPoly(x_data, y_datas, degrees)

# store mseTable in a CSV file
# define header row
header = ['degree', 'alpha_0.5', 'alpha_0.6', 'alpha_0.7', 'alpha_0.75', 'alpha_0.8', 'alpha_0.85', 'alpha_0.9', 'alpha_0.95']

# create a csv file
with open('mseTable.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(mseTable)

### Compare m_t for each degree

- for each degree, for each alpha, calculate the final m_t value
- compare the m_t values for each degree

In [164]:
def calculate_m_t(N, m0, t, poly_func):
    m = m0
    
    for i in range(1, t + 1):# iterate through the column
        # calculate k -> k has to be at least 1
        k = max(poly_func(i), 1)
        
        # Calculate E1 and E2
        E1 = (1-1/N) ** m
        E2 = (1-2/N) ** m
        average = N*(1 - E1)
        
        # check if the variance is positive, sometimes error happens because of float calculation on very small values
        if N*((N-1)*E2 + E1 - N*E1**2)<0:
            # if it is negative (error), set it to 0
            variance = 0
        else:
            # count the statistical variance of m_j+1 otherwise
            variance = sqrt(N*((N-1)*E2 + E1 - N*E1**2))
            
        # find the m_j+1
        m = average + NormalDist().inv_cdf((k - pi / 8) / (k - pi / 4 + 1)) * variance
        
    # return the final m_t
    return m

In [165]:
def get_m_t():
    alphas = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
    degrees = [2, 3, 4, 5, 6]
    
    N = 2 ** 20
    p = 1-e**-2 # our table coverage
    t = round(log(1-p)/log(1-N**(-1/3))) # Calculate t
    mt_target = N**(2/3) # our target mt
    
    mtTable = []
    
    # for each degree
    for degree in degrees:
        mtTable.append([degree])
        # for each alpha
        for i, y_data in enumerate(y_datas):
            # get m_0
            m_0 = round(mt_target/(1-alphas[i]))
            # get the polynomial fit
            poly_func, poly_pred = poly(x_data, y_data, degree)
            # calculate the m_t
            m_t = calculate_m_t(N, m_0, t , poly_func)
            # append to the table
            mtTable[-1].append(m_t)
            
    return mtTable

In [166]:
mtTable = get_m_t()

# store mtTable in a CSV file
# define header row
header = ['degree', 'alpha_0.5', 'alpha_0.6', 'alpha_0.7', 'alpha_0.75', 'alpha_0.8', 'alpha_0.85', 'alpha_0.9', 'alpha_0.95']

# create a csv file
with open('mtTable.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(mtTable)

In [167]:
for result in results:
    print(result[3][-1])

8339.804541302941
8914.040844513402
9574.379046679082
9943.512057729198
10343.190363937356
10778.579818481783
11256.969486683016
11791.967667513165
